In [1]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

### Probability questions

In [6]:
def num_strawberry():
  flavor_chances = make_array(0.3, 0.1, 0.6)
  return sample_proportions(5, flavor_chances).item(1) * 5

In [7]:
num_strawberry()

0.0

In [9]:
outcomes = make_array()
for i in range(1000):
  one_sim = num_strawberry()
  if one_sim >= 1:
    outcomes = np.append(outcomes, -5)
  else:
    outcomes = np.append(outcomes, 30-5)

net_campaign = np.sum(outcomes)

# Cookie Monster — Coding Portions

In [11]:
# Weekdays are ordered Monday through Friday.
cookies = make_array(0.18, 0.18, 0.2, 0.14, 0.3)


## Part 3: Compute TVD

Complete `get_tvd` so that it returns the total variation distance between two arrays of weekday proportions.


In [12]:
def get_tvd(observed_dist, null_dist):
    return sum(abs(observed_dist - null_dist)) / 2
observed_tvd = get_tvd(cookies,  make_array(0.2, 0.2, 0.2, 0.2, 0.2))
observed_tvd

0.10000000000000001

## Part 4: Simulated p-value

Simulate the test statistic under the null hypothesis 5,000 times. Keep track of the number of simulated TVDs at least as large as the observed TVD. Complete blanks `[A]`, `[B]`, and `[C]` below. Assume `get_tvd` is correctly implemented.


In [13]:
model = make_array(0.2, 0.2, 0.2, 0.2, 0.2)
count = 0
for i in np.arange(5000):
    sim_props = sample_proportions(150, model)
    one_sim = get_tvd(sim_props, model)
    if one_sim >= observed_tvd:
        count += 1
p = count / 5000
p


0.091

# A Tale of Two Test Statistics

In [ ]:
# seed, city_size, country_size
# 2, 70, 50: sarah reject, lena fail to reject
seed = 2
rng = np.random.default_rng(seed)
np.random.seed(seed) # sets sample()

city_lam = 2
city_size = 70
country_lam = 2.5
country_size = 70

country_pets = rng.poisson(lam=city_lam, size=city_size)
city_pets = rng.poisson(lam=country_lam, size=country_size)

pets = Table().with_columns(
  'Location', np.append(['Urban'] * city_size, ['Rural'] * country_size),
  'Pets', np.append(country_pets, city_pets)
)
pets = pets.sample(with_replacement=False) # shuffle labels
pets.show(5)
int_bins = np.arange(-0.5, 10, 1)
pets.hist(group='Location', bins=int_bins)

In [ ]:
def compute_stat(tbl, is_sarah):
    fn = np.count_nonzero
    if is_sarah:
        fn = np.mean
    grouped = tbl.group('Location', fn)
    diff = grouped.column(1).item(0) - grouped.column(1).item(1)
    if is_sarah:
        return diff
    else:
        return diff/(tbl.num_rows/2)

observed_sarah = compute_stat(pets, True)
observed_lena = compute_stat(pets, False)
observed_sarah, observed_lena

In [ ]:
def one_sim(is_sarah):
    shuffled_labels = pets.sample(with_replacement=False).column("Residence")
    shuffled_attempts = pets.with_columns("Residence",shuffled_labels)
    stat = compute_stat(shuffled_attempts, is_sarah)
    return stat

one_sim(False)

In [ ]:
nsim = 10000

stats_lena = make_array()
stats_sarah = make_array()
for i in range(nsim):
  stats_lena = np.append(stats_lena, one_sim(False))
  stats_sarah = np.append(stats_sarah, one_sim(True))

len(stats_lena), len(stats_sarah)

In [ ]:
p_value = np.count_nonzero(stats_sarah >= observed_sarah)/len(stats_sarah)
print('observed', observed_sarah)
display(p_value)
p_value < 0.05 # reject null

# Just Five More Minutes

In [ ]:
rng = np.random.default_rng(60)
np.random.seed(8)

lam, size = 3, 800
minutes = rng.poisson(lam=lam, size=size*2) * 15
weekends = ["Saturday"] * size + ["Sunday"] * size

years = np.random.choice(make_array('Freshman', 'Sophomore', 'Junior', 'Senior'), size)
ids = np.random.choice(np.arange(1000, 10000), size=size, replace=False)

students = Table().with_columns('ID', ids, 'Year', years).sort('ID')
responses = Table().with_columns('ID', np.append(ids, ids), 'Minutes', minutes, 'Day', weekends).sort('ID')

In [ ]:
students.show(3)

In [ ]:
responses.show(4)

In [ ]:
avg_per_year = responses.join('ID', students).group('Year', np.average)#.drop('ID average', 'Day average')
avg_per_year

In [ ]:
weekends = responses.pivot('Day', 'ID', 'Minutes', np.sum)
weekends.show(2)

In [ ]:
totals = weekends.column('Saturday') + weekends.column('Sunday')
totals = Table().with_column('Minutes', totals)
totals.show(3)

In [ ]:
def high_usage(value):
  return value >= 120

tiktoks = Table().with_column('High', totals.apply(high_usage, 'Minutes'))
tiktoks.show(3)

### Rewrite bootstrap

In [ ]:
np.random.seed(8)
props = make_array()

for i in np.arange(10000):
    resample = tiktoks.sample(800, with_replacement=True)
    #new_prop = np.mean(resample.column("High"))
    new_prop = np.count_nonzero(resample.column('High'))/resample.num_rows
    props = np.append(props, new_prop)
Table().with_column('Proportions', props).hist('Proportions')


In [ ]:
lower = percentile(10, props)
upper = percentile(90, props)

Table().with_column('Proportions', props).where('Proportions', are.between(lower, upper)).num_rows/len(props)